# Group 13 — Phase 3 (Step 3): Full Metric Panel, Clean T4 Latency & Patch-Occlusion Fidelity

**Repository:** `https://github.com/Uyhuynh23/Group13_CV` (branch `main`)  
**Runtime:** Google Colab **T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`)  
**Drive Data:** `/content/drive/MyDrive/Group13_CV_Data`

## Step 1: Mount Google Drive & Verify T4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## Step 2: Clone/Pull GitHub Repo (`main`) & Install Dependencies

In [ ]:
import os
if not os.path.exists('/content/Group13_CV'):
    !git clone -b main https://github.com/Uyhuynh23/Group13_CV.git /content/Group13_CV
%cd /content/Group13_CV
!git pull origin main
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/openai/CLIP.git

## Step 3: Clean T4 Latency Re-benchmark (`baseline` & `condD` seed 123)

In [ ]:
DRIVE_DATA = '/content/drive/MyDrive/Group13_CV_Data'
CACHE_PATH = f'{DRIVE_DATA}/cache/test_cache.pt'
CKPT_DIR = f'{DRIVE_DATA}/checkpoints'
PHASE3_OUT = f'{DRIVE_DATA}/phase3_results'
!mkdir -p {PHASE3_OUT}

!python evaluate.py --checkpoint {CKPT_DIR}/baseline_global_linear_seed42_best.pt --cache_path {CACHE_PATH} --out_dir {PHASE3_OUT} --beam_size 5
!python evaluate.py --checkpoint {CKPT_DIR}/condD_query_resampler_lora_seed123_best.pt --cache_path {CACHE_PATH} --out_dir {PHASE3_OUT} --beam_size 5
!python evaluate.py --checkpoint {CKPT_DIR}/condD_query_resampler_lora_seed123_best.pt --cache_path {CACHE_PATH} --out_dir {PHASE3_OUT} --beam_size 1

## Step 4: Full Metric Panel (METEOR, SPICE, CLIPScore `ViT-B/16`)
Note: If raw Flickr8k images are in `/content/drive/MyDrive/Group13_CV_Data/flickr8k_zips`, unzip them to `/content/flickr8k_raw` first for `--compute_clipscore`.

In [ ]:
import glob, os

RAW_DATA_DIR = '/content/flickr8k_raw'
if not os.path.exists(f'{RAW_DATA_DIR}/Flicker8k_Dataset'):
    !mkdir -p {RAW_DATA_DIR}
    !unzip -q -o {DRIVE_DATA}/flickr8k_zips/Flickr8k_Dataset.zip -d {RAW_DATA_DIR}

ckpts = sorted(glob.glob(f'{CKPT_DIR}/*_best.pt'))
print(f'Found {len(ckpts)} best checkpoints.')
for ckpt in ckpts:
    print('Evaluating:', os.path.basename(ckpt))
    !python evaluate.py --checkpoint {ckpt} --cache_path {CACHE_PATH} --data_dir {RAW_DATA_DIR} --out_dir {PHASE3_OUT}/full_metrics --beam_size 5 --compute_spice --compute_clipscore

## Step 5: Patch-Occlusion Fidelity ($\Delta \log p(w)$) on 100-Image Audit Subset

In [ ]:
import json, random, torch
import numpy as np
from transformers import GPT2Tokenizer
from models import build_model_from_checkpoint

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_cache = torch.load(CACHE_PATH, map_location='cpu')
with open('/content/Group13_CV/phase3_outputs/rome_audit_blinded.json') as f:
    audit_items = json.load(f)
audit_ids = [x['image_id'] for x in audit_items]

id_to_idx = {img_id: i for i, img_id in enumerate(test_cache['image_ids'])}
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

def run_occlusion(ckpt_path, k_mask=10, seed=42):
    rng = random.Random(seed)
    model, _ = build_model_from_checkpoint(ckpt_path, device=device)
    model.eval()
    drops = {'top': [], 'low': [], 'random': []}
    for img_id in audit_ids:
        idx = id_to_idx[img_id]
        patches = test_cache['patch_features'][idx].unsqueeze(0).to(device, dtype=torch.float32)
        with torch.no_grad():
            # Baseline log-prob of first generated content token
            prefix = model.connector(patches)
            out = model.gpt2(inputs_embeds=prefix, output_attentions=True)
            next_logits = out.logits[:, -1, :]
            target_tok = next_logits.argmax(dim=-1)
            base_lp = torch.log_softmax(next_logits, dim=-1)[0, target_tok[0]].item()
            # Patch importance via L2 norm of patch-to-prefix gradient/activation proxy
            scores = patches.norm(dim=-1).squeeze(0).cpu().numpy()
            ranked = np.argsort(scores)[::-1].tolist()
            top_idx = ranked[:k_mask]
            low_idx = ranked[-k_mask:]
            rand_idx = rng.sample(range(49), k_mask)
            for mode, mask_indices in [('top', top_idx), ('low', low_idx), ('random', rand_idx)]:
                masked = patches.clone()
                masked[:, mask_indices, :] = 0.0
                m_out = model.gpt2(inputs_embeds=model.connector(masked))
                m_lp = torch.log_softmax(m_out.logits[:, -1, :], dim=-1)[0, target_tok[0]].item()
                drops[mode].append(base_lp - m_lp)
    return {m: {'mean_drop': float(np.mean(v)), 'std': float(np.std(v))} for m, v in drops.items()}

occ_results = {}
for name in ['condA_patch_mlp_frozen_seed42_best.pt', 'condB_query_resampler_frozen_seed42_best.pt', 'condD_query_resampler_lora_seed42_best.pt']:
    occ_results[name] = run_occlusion(f'{CKPT_DIR}/{name}')
    print(name, occ_results[name])

with open(f'{PHASE3_OUT}/occlusion_fidelity.json', 'w') as f:
    json.dump(occ_results, f, indent=2)
print('Saved occlusion_fidelity.json!')